# W-State Routing Comparison: IQM Qubit Selector vs Beam-Search Chain Routing

This notebook runs the **same W-state preparation circuit** through two routing strategies and compares:

| | **IQM Qubit Selector** | **Beam-Search Chain Routing** |
|---|---|---|
| Qubit selection | `CostEvaluator` (GATE_COST_CZ + QNDNESS) | Custom beam search (CZ × readout × T1 × T2) |
| Transpilation | Qiskit `optimization_level=3` | `routing_method='none'` (chain guarantee) |
| SWAP overhead | Possible SWAPs inserted by Qiskit | **Zero SWAPs** by construction |
| Key insight | Best *subset* of qubits for arbitrary topology | Best *path* of qubits → linear interaction graph |

**Why this matters:** The F-gate W-state circuit has a strictly linear interaction graph (each 2-qubit gate is between consecutive logical qubits). Mapping it onto a hardware *path* of the same length eliminates all SWAP overhead — at the cost of a more constrained qubit search.

In [ ]:
N_QUBITS   = 10    # W-state size (keep ≤ 20; chain routing needs a hardware path of this length)
SHOTS      = 500   # shots per circuit
BEAM_WIDTH = 100   # beam search width for chain routing

: 

## 1. Connect to IQM Resonance

In [1]:
import os
from pathlib import Path
from iqm.qiskit_iqm import IQMProvider

secret_path = Path.cwd() / ".secrets" / "iqm_api_key"
api_token = secret_path.read_text().strip() if secret_path.exists() \
            else input("IQM Resonance API token: ").strip()
os.environ["IQM_TOKEN"] = api_token

provider = IQMProvider("https://resonance.meetiqm.com", quantum_computer="emerald")
backend  = provider.get_backend()
print(f"Connected: {backend.name}  ({backend.num_qubits} qubits)")

d:\Documents\ETH QHackathon\iqm_qte\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


: 

## 2. Load calibration data

In [ ]:
import math
import numpy as np
from iqm.qubit_selector.qubit_selector import CalibrationDataManager

cal = CalibrationDataManager().get_calibration_fidelities(backend)

def _parse_pair(fid_dict):
    out = {}
    for k, v in fid_dict.items():
        try:
            names = eval(k)
            idxs  = tuple(backend.qubit_name_to_index(n) for n in names)
            out[frozenset(idxs)] = float(v)
        except Exception:
            pass
    return out

def _parse_qubit(raw):
    out = {}
    for name, v in raw.items():
        try:
            out[backend.qubit_name_to_index(name)] = float(v)
        except Exception:
            pass
    return out

cz_fid  = _parse_pair(cal.get("CZ", {}))
ro_fid  = _parse_qubit(cal.get("readout", {}))
t1_us   = _parse_qubit(cal.get("t1", {}))
t2_us   = _parse_qubit(cal.get("t2", {}))

# Estimate W-state circuit duration
cz_durations = []
if "cz" in backend.target:
    for qargs, props in backend.target["cz"].items():
        if props and props.duration:
            cz_durations.append(props.duration)
mean_cz_ns = np.mean(cz_durations) * 1e9 if cz_durations else 80.0
circuit_ns  = 3 * (N_QUBITS - 1) * mean_cz_ns

def t1_survival(phys, wait_ns, default_t1_us=200.0):
    return float(np.exp(-wait_ns / (t1_us.get(phys, default_t1_us) * 1e3)))

def t2_survival(phys, wait_ns, default_t2_us=100.0):
    return float(np.exp(-wait_ns / (t2_us.get(phys, default_t2_us) * 1e3)))

print(f"CZ fidelity edges:  {len(cz_fid)}")
print(f"Readout fid qubits: {len(ro_fid)}")
print(f"Mean CZ duration:   {mean_cz_ns:.1f} ns")
print(f"W{N_QUBITS} circuit duration estimate: {circuit_ns:.0f} ns  ({circuit_ns/1000:.2f} µs)")

## 3. Build the W-state circuit (F-gate / Diker method)

Both approaches use **identical circuits** — the only difference is the qubit mapping.

The F-gate interaction graph is strictly **linear**: gate $k$ connects logical qubits $k-1$ and $k$.  
This is the key property our chain routing exploits.

In [ ]:
from qiskit import QuantumCircuit
import numpy as np

def _f_gate(qc, ctrl, tgt, theta):
    qc.ry(-theta, tgt)
    qc.cz(ctrl, tgt)
    qc.ry(theta, tgt)

def build_w_state_n(n):
    qc = QuantumCircuit(n, name=f"W{n}")
    qc.x(0)
    for k in range(1, n):
        theta = np.arccos(np.sqrt(1.0 / (n - k + 1)))
        _f_gate(qc, k - 1, k, theta)
    for k in range(n - 1):
        qc.cx(k + 1, k)
    return qc

qc_w = build_w_state_n(N_QUBITS)
print(f"W{N_QUBITS} circuit: depth={qc_w.depth()}, ops={dict(qc_w.count_ops())}")
print(f"Two-qubit gate pairs (logical): {[(k-1,k) for k in range(1,N_QUBITS)] + [(k+1,k) for k in range(N_QUBITS-1)]}")

## 4. Approach A — IQM Qubit Selector

The IQM Qubit Selector evaluates qubit *subsets* using calibration data and returns the lowest-cost layout.  
Qiskit then routes the circuit onto that layout with `optimization_level=3` — it may insert SWAP gates.

In [ ]:
from iqm.qubit_selector.qubit_selector import (
    CostEvaluator, CostFunction, ReadoutMode,
)
from qiskit import transpile

qc_w_measured = build_w_state_n(N_QUBITS)
qc_w_measured.measure_all()

layouts_iqm, costs_iqm = CostEvaluator(
    backend=backend,
    quantum_circuit=qc_w_measured,
    cost_function=CostFunction.GATE_COST_CZ,
    readoutmode=ReadoutMode.QNDNESS,
    num_trials=2000,
).get_top_layouts(num_layouts=10)

iqm_layout = list(layouts_iqm[0])
iqm_names  = [backend.index_to_qubit_name(q) for q in iqm_layout]
iqm_cost   = costs_iqm[0]

print(f"IQM Selector — top 5 layouts (cost = lower is better):")
for rank, (lay, cost) in enumerate(zip(layouts_iqm[:5], costs_iqm[:5]), 1):
    names = [backend.index_to_qubit_name(q) for q in lay]
    print(f"  Rank {rank}: {names}  cost={cost:.4f}")
print(f"\n✓ Selected: {iqm_names}  (cost={iqm_cost:.4f})")

# Build Z and X circuits, transpile with full Qiskit routing
qc_z_iqm = build_w_state_n(N_QUBITS); qc_z_iqm.measure_all()
qc_x_iqm = build_w_state_n(N_QUBITS)
qc_x_iqm.h(range(N_QUBITS)); qc_x_iqm.measure_all()

qc_z_iqm_t = transpile(qc_z_iqm, backend=backend, initial_layout=iqm_layout, optimization_level=3)
qc_x_iqm_t = transpile(qc_x_iqm, backend=backend, initial_layout=iqm_layout, optimization_level=3)

iqm_z_depth = qc_z_iqm_t.depth()
iqm_x_depth = qc_x_iqm_t.depth()
iqm_z_swaps = qc_z_iqm_t.count_ops().get("swap", 0)
iqm_x_swaps = qc_x_iqm_t.count_ops().get("swap", 0)

print(f"\nIQM Selector — transpiled circuit stats:")
print(f"  Z-basis: depth={iqm_z_depth}, SWAPs={iqm_z_swaps}")
print(f"  X-basis: depth={iqm_x_depth}, SWAPs={iqm_x_swaps}")

## 5. Approach B — Beam-Search Chain Routing

Because the W-state circuit's interaction graph is linear, we can search for a hardware **path** of length N.  
Mapping the circuit onto this path allows transpilation with `routing_method='none'` — **zero SWAPs guaranteed**.

Score each path: $\sum_i \log(\text{CZ}_i) + \log(\text{RO}_i) + \log(e^{-t_i/T_1}) + \log(e^{-t_i/T_2})$

In [ ]:
all_edges = list(backend.coupling_map.get_edges())
adj = {}
for u, v in all_edges:
    adj.setdefault(u, set()).add(v)
    adj.setdefault(v, set()).add(u)

def beam_search_chain(n, adj, cz_fid, ro_fid, t1_fn, t2_fn,
                      gate_ns, beam_width,
                      default_cz=0.90, default_ro=0.95):
    def ls(x):
        return math.log(max(x, 1e-12))

    beam = [(ls(ro_fid.get(q, default_ro))
             + ls(t1_fn(q, (n-1)*3*gate_ns))
             + ls(t2_fn(q, (n-1)*3*gate_ns)),
             [q])
            for q in adj]
    beam.sort(reverse=True)
    beam = beam[:beam_width]

    for step in range(n - 1):
        pos     = step + 1
        wait_ns = (n - 1 - pos) * 3 * gate_ns
        candidates = []
        for log_score, path in beam:
            last = path[-1]; used = set(path)
            for nbr in adj[last]:
                if nbr in used:
                    continue
                new_log = (log_score
                           + ls(cz_fid.get(frozenset((last, nbr)), default_cz))
                           + ls(ro_fid.get(nbr, default_ro))
                           + ls(t1_fn(nbr, wait_ns))
                           + ls(t2_fn(nbr, wait_ns)))
                candidates.append((new_log, path + [nbr]))
        if not candidates:
            break
        candidates.sort(reverse=True)
        beam = candidates[:beam_width]

    complete = [(s, p) for s, p in beam if len(p) == n]
    if not complete:
        raise RuntimeError(f"No hardware path of length {n} found — reduce N_QUBITS or increase BEAM_WIDTH")
    best_log, best_path = complete[0]
    return best_path, math.exp(best_log)

chain_layout, chain_score = beam_search_chain(
    N_QUBITS, adj, cz_fid, ro_fid,
    t1_survival, t2_survival,
    mean_cz_ns, BEAM_WIDTH,
)
chain_names = [backend.index_to_qubit_name(q) for q in chain_layout]

print(f"Chain routing — best path (combined score={chain_score:.2e}):")
print(f"  {chain_names}")
print()
print(f"{'Pos':>3}  {'QB':6}  {'Readout':>8}  {'CZ→next':>8}  {'T1(µs)':>8}  {'T2(µs)':>8}")
print("-" * 55)
for i, phys in enumerate(chain_layout):
    cz_next = cz_fid.get(frozenset((phys, chain_layout[i+1])), float("nan")) if i < N_QUBITS-1 else float("nan")
    cz_str  = f"{cz_next:.4f}" if not math.isnan(cz_next) else "  —   "
    print(f"{i:>3}  {chain_names[i]:6}  "
          f"{ro_fid.get(phys, float('nan')):>8.4f}  {cz_str:>8}  "
          f"{t1_us.get(phys, float('nan')):>8.1f}  {t2_us.get(phys, float('nan')):>8.1f}")

# Transpile with no routing — valid only because chain_layout is a hardware path
qc_z_chain = build_w_state_n(N_QUBITS); qc_z_chain.measure_all()
qc_x_chain = build_w_state_n(N_QUBITS)
qc_x_chain.h(range(N_QUBITS)); qc_x_chain.measure_all()

qc_z_chain_t = transpile(qc_z_chain, backend=backend, initial_layout=chain_layout,
                         routing_method="none", optimization_level=1)
qc_x_chain_t = transpile(qc_x_chain, backend=backend, initial_layout=chain_layout,
                         routing_method="none", optimization_level=1)

chain_z_depth = qc_z_chain_t.depth()
chain_x_depth = qc_x_chain_t.depth()
chain_z_swaps = qc_z_chain_t.count_ops().get("swap", 0)
chain_x_swaps = qc_x_chain_t.count_ops().get("swap", 0)

assert chain_z_swaps == 0 and chain_x_swaps == 0, "Unexpected SWAPs in chain routing!"
print(f"\nChain routing — transpiled circuit stats:")
print(f"  Z-basis: depth={chain_z_depth}, SWAPs={chain_z_swaps}")
print(f"  X-basis: depth={chain_x_depth}, SWAPs={chain_x_swaps}")

## 5b. Approach C — IQM Selector + Path Constraint (0 SWAPs)

Run IQM's `CostEvaluator` with a large trial budget to get the best-scored qubit *subsets*, then **filter** to the subset ranking that forms a connected hardware *path*.

| | **IQM Qubit Selector** | **IQM Selector + No SWAP** | **Beam-Search Chain** |
|---|---|---|---|
| Qubit scoring | `CostEvaluator` (CZ + QNDNESS) | `CostEvaluator` (CZ + QNDNESS) | Custom log-fidelity |
| Path constraint | ✗ (any subset) | ✓ (hardware path only) | ✓ (beam search on paths) |
| Transpilation | `optimization_level=3` | `routing_method='none'` | `routing_method='none'` |
| SWAPs | Possible | **Zero** | **Zero** |

This answers: *"Would IQM's own cost function pick better hardware paths than our hand-rolled beam search?"*

In [ ]:
## Approach C implementation — do not move this cell above the markdown header

def _is_hardware_path(layout_set, adj):
    """True iff qubit set forms a simple connected path on the hardware graph."""
    if len(layout_set) < 2:
        return True
    degree = {q: len(adj.get(q, set()) & layout_set) for q in layout_set}
    endpoints = [q for q, d in degree.items() if d == 1]
    interior  = [q for q, d in degree.items() if d == 2]
    return len(endpoints) == 2 and len(interior) == len(layout_set) - 2

def _walk_path(layout_set, adj):
    """Walk from one degree-1 endpoint to the other; returns ordered list."""
    degree = {q: len(adj.get(q, set()) & layout_set) for q in layout_set}
    start = next(q for q, d in degree.items() if d == 1)
    path = [start]
    while len(path) < len(layout_set):
        nexts = (adj.get(path[-1], set()) & layout_set) - set(path)
        if not nexts:
            break
        path.append(nexts.pop())
    return path

# Pull a large pool of IQM-scored layouts then keep only path-compatible ones
_POOL = 500
_pool_layouts, _pool_costs = CostEvaluator(
    backend=backend,
    quantum_circuit=qc_w_measured,
    cost_function=CostFunction.GATE_COST_CZ,
    readoutmode=ReadoutMode.QNDNESS,
    num_trials=8000,
).get_top_layouts(num_layouts=_POOL)

_path_candidates = [
    (_walk_path(set(lay), adj), cost)
    for lay, cost in zip(_pool_layouts, _pool_costs)
    if _is_hardware_path(set(lay), adj)
]

if not _path_candidates:
    raise RuntimeError(
        f"No path-compatible layout found among {_POOL} IQM-scored layouts. "
        "Try increasing num_trials or _POOL."
    )

iqm_path_layout, iqm_path_cost = _path_candidates[0]
iqm_path_names = [backend.index_to_qubit_name(q) for q in iqm_path_layout]

print(f"Found {len(_path_candidates)}/{_POOL} path-compatible layouts.")
print(f"Best IQM-scored path (IQM cost={iqm_path_cost:.4f}):")
print(f"  {iqm_path_names}")
print()
print(f"{'Pos':>3}  {'QB':6}  {'Readout':>8}  {'CZ→next':>8}  {'T1(µs)':>8}  {'T2(µs)':>8}")
print("-" * 55)
for i, phys in enumerate(iqm_path_layout):
    cz_next = cz_fid.get(frozenset((phys, iqm_path_layout[i+1])), float("nan")) if i < N_QUBITS-1 else float("nan")
    cz_str  = f"{cz_next:.4f}" if not math.isnan(cz_next) else "  —   "
    print(f"{i:>3}  {iqm_path_names[i]:6}  "
          f"{ro_fid.get(phys, float('nan')):>8.4f}  {cz_str:>8}  "
          f"{t1_us.get(phys, float('nan')):>8.1f}  {t2_us.get(phys, float('nan')):>8.1f}")

# Transpile with no routing — valid because iqm_path_layout is a hardware path
qc_z_iqm_path = build_w_state_n(N_QUBITS); qc_z_iqm_path.measure_all()
qc_x_iqm_path = build_w_state_n(N_QUBITS)
qc_x_iqm_path.h(range(N_QUBITS)); qc_x_iqm_path.measure_all()

qc_z_iqm_path_t = transpile(qc_z_iqm_path, backend=backend, initial_layout=iqm_path_layout,
                              routing_method="none", optimization_level=1)
qc_x_iqm_path_t = transpile(qc_x_iqm_path, backend=backend, initial_layout=iqm_path_layout,
                              routing_method="none", optimization_level=1)

iqm_path_z_depth = qc_z_iqm_path_t.depth()
iqm_path_x_depth = qc_x_iqm_path_t.depth()
iqm_path_z_swaps = qc_z_iqm_path_t.count_ops().get("swap", 0)
iqm_path_x_swaps = qc_x_iqm_path_t.count_ops().get("swap", 0)

assert iqm_path_z_swaps == 0 and iqm_path_x_swaps == 0, "Unexpected SWAPs!"
print(f"\n✓ Zero SWAPs confirmed")
print(f"  Z-basis: depth={iqm_path_z_depth}, SWAPs={iqm_path_z_swaps}")
print(f"  X-basis: depth={iqm_path_x_depth}, SWAPs={iqm_path_x_swaps}")


## 6. Pre-run comparison: depth and SWAPs

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

approaches = ["IQM Qubit\nSelector", "IQM Selector\n+ No SWAP", "Beam-Search\nChain"]
colors     = ["#e84545", "#e8a045", "#32a8a4"]
depths     = [iqm_z_depth, iqm_path_z_depth, chain_z_depth]
swaps      = [iqm_z_swaps, iqm_path_z_swaps, chain_z_swaps]

bars = axes[0].bar(approaches, depths, color=colors, width=0.5)
axes[0].set_ylabel("Circuit depth")
axes[0].set_title(f"Transpiled circuit depth (Z-basis, N={N_QUBITS})")
for bar, val in zip(bars, depths):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 str(val), ha="center", va="bottom", fontsize=13, fontweight="bold")

bars2 = axes[1].bar(approaches, swaps, color=colors, width=0.5)
axes[1].set_ylabel("SWAP gates inserted")
axes[1].set_title(f"SWAP overhead (Z-basis, N={N_QUBITS})")
axes[1].set_ylim(0, max(swaps) + 2 if max(swaps) > 0 else 3)
for bar, val in zip(bars2, swaps):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 str(val), ha="center", va="bottom", fontsize=13, fontweight="bold")

plt.suptitle("Circuit compilation comparison — before hardware execution", fontsize=12)
plt.tight_layout()
plt.show()

print(f"\n{'Metric':<30} {'IQM Selector':>15} {'IQM No-SWAP':>13} {'Chain':>8}")
print("-" * 70)
print(f"{'Z-basis depth':<30} {iqm_z_depth:>15} {iqm_path_z_depth:>13} {chain_z_depth:>8}")
print(f"{'Z-basis SWAPs':<30} {iqm_z_swaps:>15} {iqm_path_z_swaps:>13} {chain_z_swaps:>8}")
print(f"{'X-basis depth':<30} {iqm_x_depth:>15} {iqm_path_x_depth:>13} {chain_x_depth:>8}")
print(f"\nSelected qubits:")
print(f"  A IQM Selector  : {iqm_names}")
print(f"  B IQM No-SWAP   : {iqm_path_names}")
print(f"  C Chain Routing : {chain_names}")


## 7. Hardware topology: both routing strategies visualised

In [ ]:
import sys, importlib
import matplotlib.pyplot as plt

if "iqm_topology_viz" in sys.modules:
    importlib.reload(sys.modules["iqm_topology_viz"])
else:
    sys.path.insert(0, str(__import__("pathlib").Path.cwd()))
import iqm_topology_viz as tpv

# ── data format conversion ────────────────────────────────────────────────────
cz_fid_tuple = {(min(a, b), max(a, b)): v for fs, v in cz_fid.items() for a, b in [tuple(fs)]}
metrics = {
    i: {
        "readout_fidelity": ro_fid.get(i),
        "t1": t1_us[i] * 1e-6 if i in t1_us else None,
        "t2": t2_us[i] * 1e-6 if i in t2_us else None,
    }
    for i in range(backend.num_qubits)
}
pos = tpv._device_layout(backend)

# (layout, route_edges, show_arrows, title)
approaches_viz = [
    (iqm_layout,      [(iqm_layout[i],      iqm_layout[i+1])      for i in range(N_QUBITS-1)], False,
     f"A: IQM Qubit Selector\ncost={iqm_cost:.4f}  |  depth={iqm_z_depth}  |  SWAPs={iqm_z_swaps}"),
    (iqm_path_layout, [(iqm_path_layout[i], iqm_path_layout[i+1]) for i in range(N_QUBITS-1)], True,
     f"B: IQM Selector + No SWAP\ncost={iqm_path_cost:.4f}  |  depth={iqm_path_z_depth}  |  SWAPs={iqm_path_z_swaps}"),
    (chain_layout,    [(chain_layout[i],    chain_layout[i+1])    for i in range(N_QUBITS-1)], True,
     f"C: Beam-Search Chain\nscore={chain_score:.2e}  |  depth={chain_z_depth}  |  SWAPs={chain_z_swaps}"),
]

fig, axes = plt.subplots(1, 3, figsize=(33, 12))

for ax, (layout, route_edges, show_arrows, title) in zip(axes, approaches_viz):
    tpv.plot_device_topology(
        backend,
        metrics=metrics,
        cz_fidelities=cz_fid_tuple,
        color_by="readout_fidelity",
        highlight_qubits=layout,
        highlight_edges=route_edges,
        title=title,
        ax=ax,
        pos=pos,
        show_labels=True,
        spotlight=True,
    )
    # Logical qubit index badges above each highlighted node
    for logical_idx, phys in enumerate(layout):
        x, y = pos[phys]
        ax.text(
            x, y + 0.52, str(logical_idx),
            ha="center", va="bottom", fontsize=7.5, fontweight="bold",
            color="#ff3b3b", zorder=20,
            bbox=dict(boxstyle="round,pad=0.15", facecolor="white",
                      edgecolor="#ff3b3b", linewidth=1.0, alpha=0.88),
        )
    # Direction arrows on path-based approaches
    if show_arrows:
        for a, b in route_edges:
            x1, y1 = pos[a]; x2, y2 = pos[b]
            ax.annotate(
                "", xy=(x1 + 0.70*(x2-x1), y1 + 0.70*(y2-y1)),
                xytext=(x1 + 0.30*(x2-x1), y1 + 0.30*(y2-y1)),
                arrowprops=dict(arrowstyle="-|>", color="white", lw=2.0, mutation_scale=18),
                zorder=16,
            )

plt.suptitle(
    f"IQM Emerald 54-qubit chip — W{N_QUBITS} routing strategies\n"
    f"Red badges = logical qubit index 0–{N_QUBITS-1}  |  "
    f"Node colour = readout fidelity (viridis)  |  "
    f"Edge colour = CZ fidelity (RdYlGn)  |  Arrows = gate order",
    fontsize=12, y=1.01,
)
plt.tight_layout()
plt.show()


## 8. Submit jobs to hardware

Four circuits total: Z-basis and X-basis for each approach.

In [ ]:
print(f"Submitting {SHOTS} shots × 6 circuits (N={N_QUBITS})...")

job_z_iqm      = backend.run(qc_z_iqm_t,        shots=SHOTS)
job_x_iqm      = backend.run(qc_x_iqm_t,        shots=SHOTS)
job_z_iqm_path = backend.run(qc_z_iqm_path_t,   shots=SHOTS)
job_x_iqm_path = backend.run(qc_x_iqm_path_t,   shots=SHOTS)
job_z_chain    = backend.run(qc_z_chain_t,       shots=SHOTS)
job_x_chain    = backend.run(qc_x_chain_t,       shots=SHOTS)

print(f"IQM Selector      Z: {job_z_iqm.job_id()}")
print(f"IQM Selector      X: {job_x_iqm.job_id()}")
print(f"IQM Sel + No SWAP Z: {job_z_iqm_path.job_id()}")
print(f"IQM Sel + No SWAP X: {job_x_iqm_path.job_id()}")
print(f"Chain Routing     Z: {job_z_chain.job_id()}")
print(f"Chain Routing     X: {job_x_chain.job_id()}")


In [ ]:
counts_z_iqm      = job_z_iqm.result().get_counts()
counts_x_iqm      = job_x_iqm.result().get_counts()
counts_z_iqm_path = job_z_iqm_path.result().get_counts()
counts_x_iqm_path = job_x_iqm_path.result().get_counts()
counts_z_chain    = job_z_chain.result().get_counts()
counts_x_chain    = job_x_chain.result().get_counts()

print("Retrieved all 6 results.")
print(f"  IQM Selector      Z unique outcomes: {len(counts_z_iqm)}")
print(f"  IQM Sel + No SWAP Z unique outcomes: {len(counts_z_iqm_path)}")
print(f"  Chain Routing     Z unique outcomes: {len(counts_z_chain)}")


### Retrieve by job ID (if kernel was restarted)

In [ ]:
# JOB_Z_IQM      = "paste-job-id-here"
# JOB_X_IQM      = "paste-job-id-here"
# JOB_Z_IQM_PATH = "paste-job-id-here"
# JOB_X_IQM_PATH = "paste-job-id-here"
# JOB_Z_CHAIN    = "paste-job-id-here"
# JOB_X_CHAIN    = "paste-job-id-here"

# counts_z_iqm      = backend.retrieve_job(JOB_Z_IQM).result().get_counts()
# counts_x_iqm      = backend.retrieve_job(JOB_X_IQM).result().get_counts()
# counts_z_iqm_path = backend.retrieve_job(JOB_Z_IQM_PATH).result().get_counts()
# counts_x_iqm_path = backend.retrieve_job(JOB_X_IQM_PATH).result().get_counts()
# counts_z_chain    = backend.retrieve_job(JOB_Z_CHAIN).result().get_counts()
# counts_x_chain    = backend.retrieve_job(JOB_X_CHAIN).result().get_counts()


## 9. Results analysis

### 9.1 Parse counts helper

In [ ]:
def parse_counts(raw, n):
    """IQM may return space-separated registers; take the last N bits."""
    out = {}
    for bs, cnt in raw.items():
        parts = bs.strip().split()
        key   = parts[-1][-n:]
        out[key] = out.get(key, 0) + cnt
    return out

w_bitstrings = {format(1 << i, f"0{N_QUBITS}b") for i in range(N_QUBITS)}

pz_iqm      = parse_counts(counts_z_iqm,      N_QUBITS)
pz_iqm_path = parse_counts(counts_z_iqm_path, N_QUBITS)
pz_chain    = parse_counts(counts_z_chain,    N_QUBITS)
px_iqm      = parse_counts(counts_x_iqm,      N_QUBITS)
px_iqm_path = parse_counts(counts_x_iqm_path, N_QUBITS)
px_chain    = parse_counts(counts_x_chain,    N_QUBITS)

def w_fidelity(counts):
    total = sum(counts.values())
    return sum(counts.get(s, 0) for s in w_bitstrings) / total if total else 0.0

def to_probs(counts):
    total = sum(counts.values())
    return {k: v/total for k, v in counts.items()}

fid_iqm      = w_fidelity(pz_iqm)
fid_iqm_path = w_fidelity(pz_iqm_path)
fid_chain    = w_fidelity(pz_chain)

print(f"Z-basis W-state fidelity (fraction in valid single-excitation outcomes):")
print(f"  A IQM Selector:          {fid_iqm:.3f}")
print(f"  B IQM Selector + No SWAP:{fid_iqm_path:.3f}")
print(f"  C Chain Routing:         {fid_chain:.3f}")
print(f"  Ideal:                   1.000")


### 9.2 Z-basis histograms

In [ ]:
import matplotlib.pyplot as plt

def plot_z_hist(ax, counts, approach_name, shots):
    sorted_counts = sorted(counts.items(), key=lambda x: -x[1])
    labels, vals  = zip(*sorted_counts) if sorted_counts else ([], [])
    colors = ["#32a8a4" if lb in w_bitstrings else "#e84545" for lb in labels]
    ax.bar(range(len(labels)), vals, color=colors)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=90, fontsize=7)
    ax.set_ylabel("Counts")
    ax.axhline(shots/N_QUBITS, color="#888", lw=1, ls="--", label=f"ideal ({shots/N_QUBITS:.0f})")
    fid = sum(counts.get(s, 0) for s in w_bitstrings) / shots
    ax.set_title(f"{approach_name}\nW-fidelity = {fid:.3f}  (teal = valid W-state outcome)", fontsize=10)
    ax.legend(fontsize=8)

fig, axes = plt.subplots(1, 3, figsize=(22, 5))
plot_z_hist(axes[0], pz_iqm,
            f"A: IQM Qubit Selector ({SHOTS} shots, {iqm_z_swaps} SWAPs)", SHOTS)
plot_z_hist(axes[1], pz_iqm_path,
            f"B: IQM Selector + No SWAP ({SHOTS} shots, {iqm_path_z_swaps} SWAPs)", SHOTS)
plot_z_hist(axes[2], pz_chain,
            f"C: Beam-Search Chain ({SHOTS} shots, {chain_z_swaps} SWAPs)", SHOTS)
plt.suptitle(f"W{N_QUBITS} — Z-basis measurement histograms", fontsize=13)
plt.tight_layout()
plt.show()


### 9.3 X-basis entanglement witness

For the ideal $|W_N\rangle$: $\langle X_i X_j \rangle = 2/N$ for all $i \neq j$.  
For any classical mixture with the same Z marginals: $\langle X_i X_j \rangle = 0$.  
The average over all pairs is the **entanglement witness** $W$.

In [ ]:
def corr_matrix(probs, n):
    C = np.zeros((n, n))
    for i in range(n):
        for j in range(i+1, n):
            val = sum((1 - 2*int(b[-(i+1)])) * (1 - 2*int(b[-(j+1)])) * p
                      for b, p in probs.items())
            C[i, j] = C[j, i] = val
    return C

ideal_corr  = 2 / N_QUBITS
shot_stderr = 1 / np.sqrt(SHOTS)

C_iqm      = corr_matrix(to_probs(px_iqm),      N_QUBITS)
C_iqm_path = corr_matrix(to_probs(px_iqm_path), N_QUBITS)
C_chain    = corr_matrix(to_probs(px_chain),    N_QUBITS)

upper_iqm      = C_iqm[np.triu_indices(N_QUBITS, k=1)]
upper_iqm_path = C_iqm_path[np.triu_indices(N_QUBITS, k=1)]
upper_chain    = C_chain[np.triu_indices(N_QUBITS, k=1)]

W_iqm      = upper_iqm.mean()
W_iqm_path = upper_iqm_path.mean()
W_chain    = upper_chain.mean()

print(f"Entanglement witness W = mean ⟨Xi Xj⟩ over {N_QUBITS*(N_QUBITS-1)//2} pairs")
print(f"  Ideal W{N_QUBITS}:                 {ideal_corr:.4f}")
print(f"  Classical mixture:            0.0000")
print(f"  Shot noise σ:                 ±{shot_stderr:.4f}  (SHOTS={SHOTS})")
print()
print(f"  A IQM Selector:               {W_iqm:.4f}  ({W_iqm/ideal_corr*100:.1f}% of ideal, {W_iqm/shot_stderr:.1f}σ)")
print(f"  B IQM Selector + No SWAP:     {W_iqm_path:.4f}  ({W_iqm_path/ideal_corr*100:.1f}% of ideal, {W_iqm_path/shot_stderr:.1f}σ)")
print(f"  C Chain Routing:              {W_chain:.4f}  ({W_chain/ideal_corr*100:.1f}% of ideal, {W_chain/shot_stderr:.1f}σ)")


### 9.4 X-basis correlator heatmaps

In [ ]:
vmax = max(ideal_corr * 1.3, 0.01)

fig, axes = plt.subplots(1, 4, figsize=(24, 5))

def plot_corr(ax, C, title):
    im = ax.imshow(C, cmap="RdYlGn", vmin=-vmax, vmax=vmax, aspect="auto")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Qubit j"); ax.set_ylabel("Qubit i")
    ax.set_xticks(range(N_QUBITS)); ax.set_yticks(range(N_QUBITS))
    plt.colorbar(im, ax=ax, label="⟨Xi Xj⟩")

plot_corr(axes[0], C_iqm,
          f"A: IQM Qubit Selector\nW = {W_iqm:.4f}  ({W_iqm/ideal_corr*100:.0f}% of ideal)")
plot_corr(axes[1], C_iqm_path,
          f"B: IQM Selector + No SWAP\nW = {W_iqm_path:.4f}  ({W_iqm_path/ideal_corr*100:.0f}% of ideal)")
plot_corr(axes[2], C_chain,
          f"C: Beam-Search Chain\nW = {W_chain:.4f}  ({W_chain/ideal_corr*100:.0f}% of ideal)")

ideal_M = np.full((N_QUBITS, N_QUBITS), ideal_corr)
np.fill_diagonal(ideal_M, 0)
plot_corr(axes[3], ideal_M, f"Ideal W{N_QUBITS}  (2/N = {ideal_corr:.3f})")

plt.suptitle(
    f"X-basis pairwise correlators ⟨Xi Xj⟩ — W{N_QUBITS} state\n"
    f"Classical baseline = 0 everywhere; quantum ideal = {ideal_corr:.3f} off-diagonal",
    fontsize=11, y=1.02)
plt.tight_layout()
plt.show()


### 9.5 Entanglement witness bar chart

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
categories = [
    "Classical\n(separable)",
    f"A: IQM Selector\n({SHOTS} shots, {iqm_z_swaps} SWAPs)",
    f"B: IQM Sel + No SWAP\n({SHOTS} shots, 0 SWAPs)",
    f"C: Chain Routing\n({SHOTS} shots, 0 SWAPs)",
    f"Ideal W{N_QUBITS}",
]
values = [0.0, W_iqm, W_iqm_path, W_chain, ideal_corr]
colors = ["#aaaaaa", "#e84545", "#e8a045", "#32a8a4", "#1a6b69"]

bars = ax.barh(categories, values, color=colors, height=0.5)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Entanglement witness W = mean ⟨Xi Xj⟩")
ax.set_title(f"W{N_QUBITS} entanglement witness comparison")
ax.set_xlim(-0.05, ideal_corr * 1.5)
for bar, val in zip(bars, values):
    ax.text(max(val + 0.003, 0.003), bar.get_y() + bar.get_height()/2,
            f"{val:.4f}", va="center", fontsize=10)
ax.axvline(shot_stderr, color="#cc8800", lw=1, ls=":", label=f"1σ shot noise ({shot_stderr:.3f})")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


## 10. Summary

### Results table

In [ ]:
print(f"{'Metric':<35} {'A IQM Selector':>16} {'B IQM No-SWAP':>15} {'C Chain':>9} {'Ideal':>8}")
print("=" * 88)
print(f"{'Qubit scoring':<35} {'CostEvaluator':>16} {'CostEvaluator':>15} {'log-fidelity':>9} {'—':>8}")
print(f"{'Path constraint':<35} {'None':>16} {'Required':>15} {'Required':>9} {'—':>8}")
print(f"{'Transpilation':<35} {'opt_level=3':>16} {'routing=none':>15} {'routing=none':>9} {'—':>8}")
print(f"{'Z-basis circuit depth':<35} {iqm_z_depth:>16} {iqm_path_z_depth:>15} {chain_z_depth:>9} {'—':>8}")
print(f"{'Z-basis SWAP count':<35} {iqm_z_swaps:>16} {iqm_path_z_swaps:>15} {chain_z_swaps:>9} {'0':>8}")
print(f"{'Z-basis W-state fidelity':<35} {fid_iqm:>16.3f} {fid_iqm_path:>15.3f} {fid_chain:>9.3f} {'1.000':>8}")
print(f"{'Entanglement witness W':<35} {W_iqm:>16.4f} {W_iqm_path:>15.4f} {W_chain:>9.4f} {ideal_corr:>8.4f}")
print(f"{'W as % of ideal':<35} {W_iqm/ideal_corr*100:>15.1f}% {W_iqm_path/ideal_corr*100:>14.1f}% {W_chain/ideal_corr*100:>8.1f}% {'100%':>8}")
print(f"{'W in units of shot noise σ':<35} {W_iqm/shot_stderr:>15.1f}σ {W_iqm_path/shot_stderr:>14.1f}σ {W_chain/shot_stderr:>8.1f}σ {'—':>8}")
print()
print(f"Selected qubits:")
print(f"  A IQM Selector  : {iqm_names}")
print(f"  B IQM No-SWAP   : {iqm_path_names}")
print(f"  C Chain Routing : {chain_names}")
print()
print("Key insight:")
print("  B vs A — same IQM cost function, but B restricts to hardware paths → 0 SWAPs guaranteed")
print("  B vs C — same 0-SWAP guarantee, but B uses IQM's proprietary scoring vs our log-fidelity")
print(f"  If B > C in fidelity, IQM's cost function finds better paths than beam search.")
print(f"  If B ≈ C,  both scoring methods converge to the same high-quality paths.")
